# 4장 — 관성으로 달리기: 모멘텀 🎳

지그재그 문제를 재현하고, 모멘텀 한 줄(v ← β·v + g)이 그것을 어떻게 지우는지,
그리고 그 밑의 기계(EMA)의 상쇄·증폭을 숫자로 확인합니다.

**이 노트북의 목표**
1. 타원 골짜기에서 GD(지그재그) vs 모멘텀 경주 — β=0.9의 압승과 β=0.99의 진자
2. EMA의 두 얼굴 — 지그재그 입력은 상쇄, 일관 입력은 1/(1−β)배 누적
3. 보너스 — EMA가 미니배치 노이즈(3장)도 걸러 주는 것을 확인

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch4')

## 1. 실험 ① — 지그재그 vs 관성 (웹과 같은 타원 골짜기)

벽 방향 곡률 8, 전진 방향 곡률 0.1 — 같은 보폭(lr 0.2)으로 경주합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

CX, CY, LR = 0.1, 8.0, 0.2
L = lambda x, y: 0.5*CX*x*x + 0.5*CY*y*y

def run(beta, steps=60):
    """beta=0 이면 GD, 아니면 모멘텀. 난수 없음(결정적)."""
    x, y, vx, vy = -9.0, 1.5, 0.0, 0.0
    hist = [(x, y)]
    for _ in range(steps):
        vx, vy = beta*vx + CX*x, beta*vy + CY*y
        x, y = x - LR*vx, y - LR*vy
        hist.append((x, y))
    return np.array(hist)

paths = {f'β={b}': run(b) for b in [0.0, 0.9, 0.99]}

plt.figure(figsize=(8, 3.2))
th = np.linspace(0, 2*np.pi, 100)
for lv in [0.5, 2, 6, 12]:
    plt.plot(np.sqrt(2*lv/CX)*np.cos(th), np.sqrt(2*lv/CY)*np.sin(th), color='#ccc', lw=0.8)
for (name, p), c in zip(paths.items(), ['tab:blue', 'tab:orange', 'tab:red']):
    plt.plot(p[:, 0], p[:, 1], '.-', ms=3, lw=1, label=f'{name}  (L={L(*p[-1]):.3f})', color=c)
plt.scatter([0], [0], marker='*', s=150, c='gold', edgecolors='k', zorder=5)
plt.xlim(-10.5, 6); plt.ylim(-2, 2); plt.legend(); plt.grid(alpha=0.3)
plt.title('같은 보폭, 다른 기억 — 지그재그 vs 관성 vs 진자'); plt.tight_layout(); plt.show()

l_gd, l_m9, l_m99 = (L(*paths[k][-1]) for k in paths)
print(f'GD: {l_gd:.3f} / β=0.9: {l_m9:.4f} / β=0.99: {l_m99:.3f}')
assert l_m9 < l_gd / 10, 'β=0.9 모멘텀은 GD보다 10배 이상 깊이 내려간다'
assert l_m99 > l_m9 * 10, 'β=0.99는 오버슈트(진자)로 오히려 나쁘다 — 기억도 과하면 독'
print('\n✔ 확인: 관성(β=0.9)의 압승 — 단 β 과대(0.99)는 바닥을 지나치는 진자가 된다')

## 2. 실험 ② — EMA의 두 얼굴: 상쇄와 증폭

v ← β·v + g 에 두 종류의 입력을 40스텝 넣어 봅니다.

In [ ]:
def ema_final(signal, beta, n=40):
    v = 0.0
    for k in range(n):
        v = beta*v + signal(k)
    return v

print(' β      지그재그 |v|   일관된 v    증폭 이론 1/(1−β)')
for b in [0.5, 0.9, 0.95]:
    vz = abs(ema_final(lambda k: 1 if k % 2 == 0 else -1, b))
    vc = ema_final(lambda k: 1.0, b)
    print(f' {b:4.2f}   {vz:10.3f}   {vc:9.3f}   {1/(1-b):9.1f}')

vz9 = abs(ema_final(lambda k: 1 if k % 2 == 0 else -1, 0.9))
vc9 = ema_final(lambda k: 1.0, 0.9)
assert vz9 < 1.0, '지그재그(교대 부호)는 상쇄되어 입력 크기(1)보다도 작아진다'
assert abs(vc9 - 10) < 0.5, '일관 입력은 1/(1−0.9) = 10배 근처까지 쌓인다'
assert vc9 / vz9 > 15, '같은 크기 입력인데 방향의 일관성만으로 15배 이상 차이'
print('\n✔ 확인: 필터(상쇄)와 부스터(누적)는 같은 기계 — 화살표 덧셈(1부 2장)의 두 얼굴')

## 3. 실험 ③ — 보너스: EMA는 노이즈 필터이기도 하다

3장의 흔들리는 나침반(일관된 기울기 + 노이즈)을 EMA에 통과시켜 봅니다.

In [ ]:
rng = np.random.default_rng(404)
noisy = 1.0 + 1.5 * rng.standard_normal(2000)      # 진짜 기울기 1 + 노이즈 σ=1.5

beta = 0.9
v, vs = 0.0, []
for g in noisy:
    v = beta*v + (1-beta)*g                         # 크기 보존형 EMA (평균 꼴)
    vs.append(v)
vs = np.array(vs[100:])                             # 워밍업 구간 제외

print(f'원래 신호의 표준편차: {noisy.std():.3f}')
print(f'EMA 통과 후 표준편차: {vs.std():.3f}  (평균 {vs.mean():.3f} — 진짜 기울기 1 근처)')
assert vs.std() < noisy.std() / 3, 'EMA가 흔들림을 크게 줄인다 — 모멘텀은 노이즈 필터이기도'
assert abs(vs.mean() - 1.0) < 0.1, '평균(진짜 방향)은 보존된다'
print('\n✔ 확인: 지그재그 상쇄와 같은 원리로 미니배치 노이즈도 매끈해진다 (3장과의 시너지)')

## 4. 관찰 정리 + 🤖 AI 연결

- **지그재그 문제** = 축별 곡률 차 + 단일 보폭(2장의 딜레마) — 벽 방향은 튕기고 전진은 굼뜸.
- **모멘텀** v ← β·v + g: 진동은 상쇄, 전진은 1/(1−β)배 누적 — β=0.9면 10배 부스터.
  과하면(0.99) 바닥을 지나치는 진자(오버슈트).
- 밑의 기계는 **EMA**(최근 크게·과거는 β^k로 잊는 평균, 기억 ≈ 1/(1−β)걸음) —
  같은 원리로 미니배치 노이즈도 걸러 준다. 정체는 벡터 덧셈의 상쇄·누적(1부 2장).
- 실무: momentum=0.9, Adam의 β₁ — 오늘 만든 것이 Adam의 첫 부품.

**다음 챕터**: 5장 — Adam (핵심). 진동은 지웠지만 보폭은 여전히 하나 —
축마다 보폭을 자동으로 맞추는 두 번째 EMA(β₂)를 붙여 현대 훈련의 기본기를 조립합니다.